# 🔍 PCB Defect & Component Detection: Ground Truth vs Model Predictions

This notebook allows interactive comparison of **Ground Truth** annotations against **Model Predictions** across our benchmarked and novel YOLO models.
- Compare any trained model (`yolov11s`, `yolov26s_loss_reweight`, `yolov26s_bbox_cappaste`, `yolov9s`, etc.)
- **SAHI (Slicing Aided Hyper Inference)** toggle: Resolves the small-component resolution bottleneck for micro 0402 SMD chip capacitors without retraining
- Side-by-side visualization with distinct colors for the 4 target classes:
  - **Capacitor** (Cyan)
  - **Connector** (Gold / Yellow)
  - **Electrolytic Capacitor** (Magenta)
  - **IC** (Lime Green)
- Inspect confidence thresholds, IoU overlap, and missed detections.

In [1]:
# 1. Setup & Imports
import os
import glob
import sys
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torchvision
from ultralytics import YOLO

# Ensure fresh reload of custom modules without stale Jupyter cache
if 'tools.sahi_pcb_inference' in sys.modules:
    del sys.modules['tools.sahi_pcb_inference']

# Import SAHI Hyper-Inference & DIoU-NMS (Zheng et al., AAAI 2020) modules
from tools.sahi_pcb_inference import (
    sahi_predict,
    diou_nms,
    batched_diou_nms,
    soft_nms,
    evaluate_sahi_benchmark,
    draw_detections
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 150
print("Libraries, SAHI, and DIoU-NMS modules successfully imported!")


ModuleNotFoundError: No module named 'tools.sahi_pcb_inference'

In [ ]:
# 2. Define Target Classes and Visual Styles
RAW_CLASSES = [
    "Button", "Capacitor Jumper", "Capacitor", "Clock", "Connector",
    "Diode", "EM", "Electrolytic Capacitor", "Ferrite Bead", "IC",
    "Inductor", "Jumper", "Led", "Pads", "Pins", "Resistor",
    "Resistor Jumper", "Resistor Network", "Switch", "Test Point",
    "Transistor", "Unknown", "Variable Resistor"
]

# Evaluated 4 target classes
EVAL_CLASSES = {0: "Capacitor", 1: "Connector", 2: "Electrolytic Capacitor", 3: "IC"}

CLASS_COLORS = {
    "Capacitor": (0, 220, 255),               # Cyan
    "Connector": (255, 190, 0),               # Gold
    "Electrolytic Capacitor": (255, 50, 180), # Magenta
    "IC": (0, 255, 100),                      # Bright Green
    "Other": (180, 180, 180)                  # Light Gray
}

def get_color(class_name):
    return CLASS_COLORS.get(class_name, CLASS_COLORS["Other"])

In [ ]:
# 3. Discover Available Trained Checkpoints
weight_files = sorted(glob.glob("runs/**/best.pt", recursive=True))

print(f"Found {len(weight_files)} trained models locally:")
for idx, wf in enumerate(weight_files):
    model_name = wf.split('/')[1]
    print(f"  [{idx}] {model_name:35s} -> {wf}")

In [ ]:
# 4. Helper Functions to Draw Ground Truth and Predictions
def load_ground_truth(txt_path, img_w, img_h, map_capacitor_jumpers=True):
    """
    Load ground truth bounding boxes from YOLO annotation text file.
    Auto-detects whether the file is in Unified 4-Class or Legacy 23-Class format.
    """
    boxes = []
    if not os.path.exists(txt_path):
        return boxes
    raw_lines = []
    with open(txt_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                raw_lines.append((int(parts[0]), [float(p) for p in parts[1:5]]))

    if not raw_lines:
        return boxes

    # If any class index is >= 4, it is definitely a Legacy 23-class file
    is_legacy = any(cid >= 4 for cid, _ in raw_lines)

    for cid, (cx, cy, w, h) in raw_lines:
        x1 = int((cx - w / 2) * img_w)
        y1 = int((cy - h / 2) * img_h)
        x2 = int((cx + w / 2) * img_w)
        y2 = int((cy + h / 2) * img_h)

        if is_legacy:
            cap_ids = (1, 2) if map_capacitor_jumpers else (2,)
            if cid in cap_ids:
                cls_name = "Capacitor"
                norm_cid = 0
            elif cid == 4:
                cls_name = "Connector"
                norm_cid = 1
            elif cid == 7:
                cls_name = "Electrolytic Capacitor"
                norm_cid = 2
            elif cid in (9, 22):
                cls_name = "IC"
                norm_cid = 3
            else:
                continue
        else:
            if cid in EVAL_CLASSES:
                norm_cid = cid
                cls_name = EVAL_CLASSES[cid]
            else:
                continue

        boxes.append((norm_cid, cls_name, x1, y1, x2, y2))
    return boxes

def draw_boxes(image, boxes, is_prediction=False, filter_target_only=True):
    img = image.copy()
    h, w = img.shape[:2]
    for b in boxes:
        if is_prediction:
            cls_id, cls_name, conf, x1, y1, x2, y2 = b
            canonical_name = "Capacitor" if cls_name in ("Capacitor", "Capacitor Jumper") else cls_name
            label = f"{canonical_name} {conf:.2f}"
        else:
            cls_id, cls_name, x1, y1, x2, y2 = b
            canonical_name = "Capacitor" if cls_name in ("Capacitor", "Capacitor Jumper") else cls_name
            label = canonical_name

        # Highlight evaluated target classes
        if filter_target_only:
            if canonical_name not in EVAL_CLASSES.values():
                continue

        color = get_color(canonical_name)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

        # Label background
        y_label = max(y1 - 6, 12)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(img, (x1, y_label - th - 2), (x1 + tw + 2, y_label + 2), color, -1)
        cv2.putText(img, label, (x1 + 1, y_label), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 1, cv2.LINE_AA)
    return img


In [ ]:
# 5. Select Model and Run Inference
CHOSEN_MODEL = "superyolo26s_rectified_1280"

# --- Threshold & Resolution ---
CONF_THRESHOLD = 0.15      # 0.15 for micro-capacitors | 0.25 for standard | 0.05 for high recall
IMGSZ = 1280               # 1280 for superyolo_1280 | 640 for superyolo_640

# --- Standard YOLO vs SAHI ---
USE_SAHI = False           # False: test pure SuperYOLO model (~58 FPS)
                           # True:  test Slicing Aided Hyper-Inference
SLICE_SIZE = 480
OVERLAP_RATIO = 0.25
IOU_THRESHOLD = 0.45
NMS_TYPE = "diou"
BETA = 1.0

# --- Robust Path Detection (works for both new & older runs) ---
model_path = f"runs/{CHOSEN_MODEL}/weights/best.pt"
if not os.path.exists(model_path):
    model_path = f"runs/{CHOSEN_MODEL}/pcb-filtered/weights/best.pt"

if not os.path.exists(model_path):
    avail = [p.split("/")[1] for p in sorted(glob.glob("runs/**/best.pt", recursive=True))]
    raise FileNotFoundError(f"Model '{CHOSEN_MODEL}' not found. Available models:\n  " + "\n  ".join(avail))

print(f"Loading model: {model_path}...")
model = YOLO(model_path)
print("Model loaded successfully!")
print(f"Inference Mode: {'🚀 SAHI Hyper-Inference [' + NMS_TYPE.upper() + '-NMS] (Slice=' + str(SLICE_SIZE) + 'px)' if USE_SAHI else '⚡ Standard YOLO (' + str(IMGSZ) + 'px)'}")


In [ ]:
# 6. Visual Comparison Gallery (Ground Truth vs Model Predictions)
# Load genuine native-resolution test images (2048x1365):
native_images = sorted(glob.glob("data_samples/native_images/*.jpg"))
sample_images = native_images[:6] if native_images else sorted(glob.glob("data_samples/images/*.jpg"))[:6]

if not sample_images:
    print("❌ No images found!")
else:
    mode_str = f"SAHI Hyper-Inference (slice={SLICE_SIZE}px)" if USE_SAHI else f"Standard YOLO ({IMGSZ}px)"
    print(f"🔍 Displaying {len(sample_images)} test PCB samples for model: {CHOSEN_MODEL} [{mode_str}]")

for img_path in sample_images:
    raw_img = cv2.imread(img_path)
    if raw_img is None:
        continue
    rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    h, w = rgb_img.shape[:2]
    
    # 1. Load Ground Truth boxes (automatically checks native_labels first)
    lbl_path = img_path.replace("native_images", "native_labels").replace("images", "labels").replace(".jpg", ".txt")
    gt_boxes = load_ground_truth(lbl_path, w, h)
    gt_vis = draw_boxes(rgb_img, gt_boxes, is_prediction=False)
    target_gt = [b for b in gt_boxes if b[1] in EVAL_CLASSES.values() or b[0] in EVAL_CLASSES]
    
    # 2. Run Model Inference
    if USE_SAHI:
        pred_boxes = sahi_predict(
            model, raw_img,
            slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
            overlap_ratio=OVERLAP_RATIO, conf_threshold=CONF_THRESHOLD,
            iou_threshold=IOU_THRESHOLD, nms_type=NMS_TYPE, beta=BETA
        )
    else:
        results = model(raw_img, imgsz=IMGSZ, conf=CONF_THRESHOLD, verbose=False)
        pred_boxes = []
        for box in results[0].boxes:
            cid = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
            cname = model.names.get(cid, str(cid))
            pred_boxes.append((cid, cname, conf, x1, y1, x2, y2))
        
    pred_vis = draw_boxes(rgb_img, pred_boxes, is_prediction=True)
    target_preds = [b for b in pred_boxes if b[1] in EVAL_CLASSES.values() or b[0] in EVAL_CLASSES]
    
    # 3. Render Side-by-Side
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(gt_vis)
    axes[0].set_title(f"Ground Truth ({len(target_gt)} target objs) | {Path(img_path).name[:30]}", fontsize=12, fontweight='bold')
    axes[0].axis("off")
    
    mode_tag = f"SAHI [{NMS_TYPE.upper()}]" if USE_SAHI else "Standard"
    axes[1].imshow(pred_vis)
    axes[1].set_title(f"{CHOSEN_MODEL} [{mode_tag}] ({len(target_preds)} target detected | conf >= {CONF_THRESHOLD})", fontsize=12, fontweight='bold', color='darkgreen' if len(target_preds) > 0 else 'black')
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()


In [ ]:
# 7. Compare Two Models Side-by-Side (e.g. Baseline vs Improved Technique)
def compare_two_models(model_a_name, model_b_name, img_path, conf=0.25):
    raw_img = cv2.imread(img_path)
    rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    
    mA = YOLO(f"runs/{model_a_name}/pcb-filtered/weights/best.pt")
    mB = YOLO(f"runs/{model_b_name}/pcb-filtered/weights/best.pt")
    
    resA = mA.predict(raw_img, imgsz=640, conf=conf, verbose=False)[0]
    resB = mB.predict(raw_img, imgsz=640, conf=conf, verbose=False)[0]
    
    boxesA = [(int(b.cls[0]), mA.names[int(b.cls[0])], float(b.conf[0]), int(b.xyxy[0][0]), int(b.xyxy[0][1]), int(b.xyxy[0][2]), int(b.xyxy[0][3])) for b in resA.boxes]
    boxesB = [(int(b.cls[0]), mB.names[int(b.cls[0])], float(b.conf[0]), int(b.xyxy[0][0]), int(b.xyxy[0][1]), int(b.xyxy[0][2]), int(b.xyxy[0][3])) for b in resB.boxes]
    
    visA = draw_boxes(rgb_img, boxesA, is_prediction=True)
    visB = draw_boxes(rgb_img, boxesB, is_prediction=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    axes[0].imshow(visA)
    axes[0].set_title(f"Model A: {model_a_name}", fontsize=12, fontweight='bold')
    axes[0].axis("off")
    
    axes[1].imshow(visB)
    axes[1].set_title(f"Model B: {model_b_name}", fontsize=12, fontweight='bold')
    axes[1].axis("off")
    
    plt.tight_layout()
    plt.show()
# Example comparison between Baseline YOLO26s and Loss Reweighted YOLO26s:
if len(sample_images) > 0 and os.path.exists("runs/yolov26s/pcb-filtered/weights/best.pt") and os.path.exists("runs/yolov26s_loss_reweight/pcb-filtered/weights/best.pt"):
    compare_two_models("yolov26s", "yolov26s_loss_reweight", sample_images[0], conf=0.25)


In [ ]:
# 8. Interactive Comparison: Ground Truth vs Standard YOLO vs SAHI Hyper-Inference
# This comparative visualizer shows how SAHI recovers small 0402 chip capacitors & adjacent connectors
def compare_groundtruth_standard_sahi(img_path, conf=0.20, slice_size=360, overlap=0.25):
    raw_img = cv2.imread(img_path)
    if raw_img is None:
        print(f"❌ Image not found: {img_path}")
        return
    rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    h, w = rgb_img.shape[:2]
    
    # 1. Ground Truth
    lbl_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
    gt_boxes = load_ground_truth(lbl_path, w, h)
    gt_vis = draw_boxes(rgb_img, gt_boxes, is_prediction=False)
    target_gt = [b for b in gt_boxes if b[1] in EVAL_CLASSES.values() or b[0] in EVAL_CLASSES]
    
    # 2. Standard YOLO (640px)
    results_std = model.predict(raw_img, imgsz=IMGSZ, conf=conf, verbose=False)[0]
    std_boxes = []
    for box in results_std.boxes:
        cls_id = int(box.cls[0])
        score = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_name = model.names[cls_id] if cls_id in model.names else RAW_CLASSES[cls_id]
        std_boxes.append((cls_id, cls_name, score, x1, y1, x2, y2))
    std_vis = draw_boxes(rgb_img, std_boxes, is_prediction=True)
    target_std = [b for b in std_boxes if b[1] in EVAL_CLASSES.values() or b[0] in EVAL_CLASSES]
    
    # 3. SAHI Hyper-Inference
    sahi_boxes = sahi_predict(
        model=model,
        image_bgr=raw_img,
        slice_height=slice_size,
        slice_width=slice_size,
        overlap_ratio=overlap,
        conf_threshold=conf,
        iou_threshold=0.45,
        include_full_image=True
    )
    sahi_vis = draw_boxes(rgb_img, sahi_boxes, is_prediction=True)
    target_sahi = [b for b in sahi_boxes if b[1] in EVAL_CLASSES.values() or b[0] in EVAL_CLASSES]
    
    # 4. Render 3-Panel Side-by-Side
    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    
    axes[0].imshow(gt_vis)
    axes[0].set_title(f"[1] Ground Truth ({len(target_gt)} target objs)\n{Path(img_path).name[:30]}", fontsize=13, fontweight='bold')
    axes[0].axis("off")
    
    axes[1].imshow(std_vis)
    axes[1].set_title(f"[2] Standard YOLO {IMGSZ}px ({len(target_std)} detected | conf>={conf})\nMisses sub-pixel 0402 capacitors", fontsize=13, fontweight='bold', color='navy')
    axes[1].axis("off")
    
    diff_count = len(target_sahi) - len(target_std)
    sign = '+' if diff_count >= 0 else ''
    axes[2].imshow(sahi_vis)
    axes[2].set_title(f"[3] SAHI Hyper-Inference ({len(target_sahi)} detected | {sign}{diff_count} micro-objs)\nSlices preserve native sensor resolution", fontsize=13, fontweight='bold', color='darkgreen')
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
# Run 3-Way Comparative Visualizer on ATTIOT Bottom PCB sample:
target_sample = "data_samples/images/ATTIOT_Bottom_jpg.rf.8a97ad6664656973c60d95057d9d473c.jpg"
if os.path.exists(target_sample):
    print("🔬 Running 3-Way Comparative Benchmark on ATTIOT Bottom PCB...")
    compare_groundtruth_standard_sahi(target_sample, conf=0.20, slice_size=360, overlap=0.25)
elif len(sample_images) > 0:
    compare_groundtruth_standard_sahi(sample_images[0], conf=0.20, slice_size=360, overlap=0.25)


### 9. 📊 Quantitative Benchmark: Compute mAP50 Before vs After SAHI

This cell quantitatively calculates **Precision, Recall, per-class AP50** (Capacitor, Connector, Electrolytic Capacitor, IC), and **overall mAP50** under the standard PASCAL VOC / COCO all-points interpolated protocol (IoU=0.50).

It runs **Standard 640px YOLO** against **SAHI Hyper-Inference** on your test PCB samples, prints a comparison table with exact $\Delta$ gains, and renders a side-by-side performance bar chart.

In [ ]:
# 9. Compute and Compare mAP50: Standard YOLO vs SAHI Hyper-Inference
benchmark_results = evaluate_sahi_benchmark(
    model=model,
    sample_dir="data_samples",
    max_images=10,         # Select number of images (e.g. 10 for quick ~15s run, or None for all 44)
    slice_size=480,        # Tile window size in pixels
    overlap=0.25,          # 25% window overlap
    conf_thresh=0.01,      # Low threshold for full Precision-Recall curve integration
    iou_thresh=0.50,       # Standard AP50 matching threshold
    imgsz=IMGSZ,
    plot=True              # Displays side-by-side bar chart with AP gains
)


### 10. 🔌 Solution: Distance-IoU NMS (DIoU-NMS) for Touching Connectors

**The Problem:** When two identical connectors sit side-by-side with 0px gap (such as the bottom dual header sockets on the ATTIOT board), standard **Hard-NMS** sees boundary proposal overlap and suppresses the second connector.

**The Solution (Zheng et al., AAAI 2020):** **DIoU-NMS** incorporates the normalized central distance between boxes:
$$\mathcal{S}_{\text{DIoU}} = \text{IoU} - \left(\frac{\rho^2(b, b_i)}{c^2}\right)^\beta$$
Because adjacent connectors have distinct physical centers ($b \neq b_i$), $\frac{\rho^2}{c^2}$ is large, which pulls $\mathcal{S}_{\text{DIoU}}$ well below the threshold $\epsilon$. **Both connectors are cleanly preserved without false suppression!**

In [ ]:
# 10. Demonstrate DIoU-NMS vs Standard Hard-NMS on Adjacent Touching Connectors
sample_img_path = "data_samples/images/DuetWIFI_Top_png_jpg.rf.bb4dd20dee68b692f60e78bd86a39b70.jpg"
if os.path.exists(sample_img_path):
    raw_test = cv2.imread(sample_img_path)
    rgb_test = cv2.cvtColor(raw_test, cv2.COLOR_BGR2RGB)
    
    # 1. Run SAHI with Standard Hard-NMS
    preds_hard = sahi_predict(model, raw_test, slice_height=480, slice_width=480, conf_threshold=0.15, nms_type='hard')
    vis_hard = draw_boxes(rgb_test, preds_hard, is_prediction=True)
    conns_hard = [d for d in preds_hard if d[1] == 'Connector']
    
    # 2. Run SAHI with DIoU-NMS (Zheng et al. AAAI 2020)
    preds_diou = sahi_predict(model, raw_test, slice_height=480, slice_width=480, conf_threshold=0.15, nms_type='diou')
    vis_diou = draw_boxes(rgb_test, preds_diou, is_prediction=True)
    conns_diou = [d for d in preds_diou if d[1] == 'Connector']
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    axes[0].imshow(vis_hard)
    axes[0].set_title(f"Standard Hard-NMS ({len(conns_hard)} Connectors Detected)\nRisk of touching object suppression", fontsize=12, fontweight='bold', color='navy')
    axes[0].axis('off')
    
    axes[1].imshow(vis_diou)
    axes[1].set_title(f"DIoU-NMS (Zheng et al. AAAI 2020) ({len(conns_diou)} Connectors Detected)\nCenters separated -> touching headers preserved!", fontsize=12, fontweight='bold', color='darkgreen')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    print(f"✅ Hard-NMS Connectors: {len(conns_hard)} | DIoU-NMS Connectors: {len(conns_diou)}")


### 11. 🔍 Side-by-Side Visual Comparison: Ground Truth vs NMS Predictions

This cell visualizes **Ground Truth (GT)** and **NMS Predictions (DIoU-NMS)** side-by-side for direct spatial comparison:
- **Left Panel:** Ground Truth labeled bounding boxes from dataset annotations.
- **Right Panel:** Model predictions after **DIoU-NMS** suppression (Zheng et al., AAAI 2020).
- **Bottom Insets:** High-resolution zoom crops centered on dense/touching components (such as adjacent pin header connectors).


In [ ]:
# 11. Side-by-Side Visual Comparison: Ground Truth vs NMS Predictions
# Choose any test PCB image:
# target_img_path = "data_samples/images/ATTIOT_Bottom_jpg.rf.8a97ad6664656973c60d95057d9d473c.jpg"
target_img_path = "data_samples/images/DuetWIFI_Top_png_jpg.rf.bb4dd20dee68b692f60e78bd86a39b70.jpg"

if not os.path.exists(target_img_path):
    avail_imgs = sorted(glob.glob("data_samples/images/*.jpg"))
    target_img_path = avail_imgs[0] if avail_imgs else None

if target_img_path and os.path.exists(target_img_path):
    raw_img = cv2.imread(target_img_path)
    rgb_img = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    h, w = rgb_img.shape[:2]
    
    # 1. Load Ground Truth Annotations (with Class 1 -> 2 Capacitor aliasing)
    lbl_path = target_img_path.replace("images", "labels").replace(".jpg", ".txt")
    gt_boxes = load_ground_truth(lbl_path, w, h)
    target_gt = [b for b in gt_boxes if b[1] in EVAL_CLASSES.values() or b[0] in EVAL_CLASSES]
    gt_vis = draw_boxes(rgb_img, gt_boxes, is_prediction=False, filter_target_only=True)
    
    # 2. Run Multi-Scale SAHI Inference with NMS (DIoU-NMS)
    nms_type_eval = "diou"   # Options: 'diou' (recommended), 'hard', or 'soft'
    conf_thresh_eval = 0.15  # Optimal sensitivity threshold
    
    nms_preds = sahi_predict(
        model=model,
        image_bgr=raw_img,
        slice_height=480,
        slice_width=480,
        overlap_ratio=0.25,
        conf_threshold=conf_thresh_eval,
        iou_threshold=0.45,
        nms_type=nms_type_eval,
        include_full_image=True
    )
    target_nms = [p for p in nms_preds if p[1] in EVAL_CLASSES.values() or p[0] in EVAL_CLASSES]
    nms_vis = draw_boxes(rgb_img, nms_preds, is_prediction=True, filter_target_only=True)
    
    # 3. Render 2-Panel Side-by-Side Plot with High-Resolution Insets
    fig = plt.figure(figsize=(20, 12))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1.4], hspace=0.18, wspace=0.06)
    
    # Top-Left: Full Ground Truth Board
    ax_gt = fig.add_subplot(gs[0, 0])
    ax_gt.imshow(gt_vis)
    ax_gt.set_title(f"[A] Ground Truth ({len(target_gt)} target objects | {len(gt_boxes)} total annotations)\n{Path(target_img_path).name[:45]}", fontsize=13, fontweight='bold', color='#1E3A8A', pad=8)
    ax_gt.axis("off")
    
    # Top-Right: Full NMS Predictions Board
    ax_nms = fig.add_subplot(gs[0, 1])
    ax_nms.imshow(nms_vis)
    ax_nms.set_title(f"[B] NMS Predictions [{nms_type_eval.upper()}-NMS] ({len(target_nms)} targets detected | conf >= {conf_thresh_eval})\nMulti-Scale SAHI Inference", fontsize=13, fontweight='bold', color='#065F46', pad=8)
    ax_nms.axis("off")
    
    # Bottom: Zoom Insets
    ymin, ymax = max(0, int(h * 0.78)), h
    xmin, xmax = int(w * 0.30), min(w, int(w * 0.95))
    crop_gt = gt_vis[ymin:ymax, xmin:xmax]
    crop_nms = nms_vis[ymin:ymax, xmin:xmax]
    
    ax_cgt = fig.add_subplot(gs[1, 0])
    ax_cgt.imshow(crop_gt)
    ax_cgt.set_title("Zoom Inset: Ground Truth", fontsize=11, fontweight='bold', color='#1E3A8A')
    ax_cgt.axis("off")
    
    ax_cnms = fig.add_subplot(gs[1, 1])
    ax_cnms.imshow(crop_nms)
    ax_cnms.set_title(f"Zoom Inset: {nms_type_eval.upper()}-NMS Predictions", fontsize=11, fontweight='bold', color='#065F46')
    ax_cnms.axis("off")
    
    plt.show()
    
    from collections import Counter
    target_counts = dict(Counter([b[1] for b in target_gt]))
    total_counts = dict(Counter([b[1] for b in gt_boxes]))
    non_target_counts = {k: v for k, v in total_counts.items() if k not in EVAL_CLASSES.values()}
    print(f"Total annotations in label file: {len(gt_boxes)}")
    print(f"Target evaluated objects ({len(target_gt)} drawn): {target_counts}")
    print(f"Non-target annotations ({sum(non_target_counts.values())} filtered, e.g. Resistors/Pins): {non_target_counts}")
    print(f"Side-by-side comparison ready: {len(target_gt)} GT targets vs {len(target_nms)} NMS detected targets.")
else:
    print(f"Image not found at {target_img_path}")



### 12. 📊 Quantitative & Detailed Results of NMS

This cell analyzes and displays the **exact numerical results** produced by NMS:
1. **Summary Table:** Direct class-by-class comparison between Ground Truth, Hard-NMS, and DIoU-NMS.
2. **Detailed Detections List:** Full table of detected components with bounding box coordinates `[x1, y1, x2, y2]`, width, height, and confidence scores.
3. **DIoU Geometric Diagnostic:** Mathematical breakdown of adjacent/touching components calculating Center Distance ($\rho$), Enclosing Diagonal ($c$), IoU, and DIoU suppression penalty $\frac{\rho^2}{c^2}$.


In [ ]:
# 12. Quantitative & Detailed Results of NMS
# Computes complete per-class statistics, detection coordinates, and DIoU geometric metrics
import pandas as pd

# 1. Run Standard Hard-NMS baseline for direct quantitative comparison
preds_hard = sahi_predict(
    model=model,
    image_bgr=raw_img,
    slice_height=480,
    slice_width=480,
    overlap_ratio=0.25,
    conf_threshold=conf_thresh_eval,
    iou_threshold=0.45,
    nms_type="hard",
    include_full_image=True
)

# 2. Build Comparative Summary Table
summary_rows = []
for cid, cname in EVAL_CLASSES.items():
    n_gt = sum(1 for b in gt_boxes if b[1] == cname)
    n_hard = sum(1 for p in preds_hard if p[1] == cname)
    c_preds = [p for p in nms_preds if p[1] == cname]
    n_nms = len(c_preds)
    confs = [p[2] for p in c_preds]
    avg_c = np.mean(confs) if confs else 0.0
    max_c = np.max(confs) if confs else 0.0
    delta = n_nms - n_hard
    delta_str = f"+{delta}" if delta > 0 else (f"{delta}" if delta < 0 else "0")
    
    summary_rows.append({
        "Class": cname,
        "Ground Truth": n_gt,
        "Hard-NMS Count": n_hard,
        "DIoU-NMS Count": n_nms,
        "Delta (DIoU - Hard)": delta_str,
        "Avg Confidence": f"{avg_c:.3f}",
        "Max Confidence": f"{max_c:.3f}"
    })

# Add TOTAL row
tot_gt = sum(r["Ground Truth"] for r in summary_rows)
tot_hard = sum(r["Hard-NMS Count"] for r in summary_rows)
tot_nms = sum(r["DIoU-NMS Count"] for r in summary_rows)
summary_rows.append({
    "Class": "TOTAL (4 Target Classes)",
    "Ground Truth": tot_gt,
    "Hard-NMS Count": tot_hard,
    "DIoU-NMS Count": tot_nms,
    "Delta (DIoU - Hard)": f"+{tot_nms - tot_hard}" if tot_nms >= tot_hard else f"{tot_nms - tot_hard}",
    "Avg Confidence": f"{np.mean([p[2] for p in target_nms]):.3f}" if target_nms else "0.000",
    "Max Confidence": f"{np.max([p[2] for p in target_nms]):.3f}" if target_nms else "0.000"
})

df_summary = pd.DataFrame(summary_rows)
print("=" * 88)
print("📊 1. NMS vs Ground Truth Comparative Summary Table")
print("=" * 88)
print(df_summary.to_string(index=False))

# 3. Build Detailed Detections Inspector Table (Top 15 highest confidence)
det_rows = []
for idx, p in enumerate(target_nms):
    cls_id, cls_name, conf, x1, y1, x2, y2 = p
    bw = x2 - x1
    bh = y2 - y1
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    det_rows.append({
        "Rank": idx + 1,
        "Class": cls_name,
        "Confidence": round(conf, 4),
        "Box [x1, y1, x2, y2]": [x1, y1, x2, y2],
        "W (px)": bw,
        "H (px)": bh,
        "Area (px²)": bw * bh,
        "Center (cx, cy)": (round(cx, 1), round(cy, 1))
    })

df_dets = pd.DataFrame(det_rows)
print("\n" + "=" * 88)
print(f"📋 2. Sample of Detected Objects after DIoU-NMS (Top 15 by Confidence)")
print("=" * 88)
print(df_dets.head(15).to_string(index=False))

# 4. Mathematical Diagnostic for Touching / Adjacent Connectors
print("\n" + "=" * 88)
print("🔬 3. Mathematical Diagnostic: Adjacent Touching Connectors")
print("=" * 88)
bottom_conns = sorted([p for p in target_nms if p[1] == "Connector" and p[4] > 500], key=lambda x: x[3])
if len(bottom_conns) >= 2:
    for i in range(len(bottom_conns) - 1):
        c1, c2 = bottom_conns[i], bottom_conns[i+1]
        b1 = [c1[3], c1[4], c1[5], c1[6]]
        b2 = [c2[3], c2[4], c2[5], c2[6]]
        
        cx1, cy1 = (b1[0] + b1[2]) / 2.0, (b1[1] + b1[3]) / 2.0
        cx2, cy2 = (b2[0] + b2[2]) / 2.0, (b2[1] + b2[3]) / 2.0
        
        rho2 = (cx1 - cx2)**2 + (cy1 - cy2)**2
        rho = np.sqrt(rho2)
        
        enc_x1, enc_y1 = min(b1[0], b2[0]), min(b1[1], b2[1])
        enc_x2, enc_y2 = max(b1[2], b2[2]), max(b1[3], b2[3])
        c2_diag = (enc_x2 - enc_x1)**2 + (enc_y2 - enc_y1)**2
        c_diag = np.sqrt(c2_diag)
        
        inter_x1 = max(b1[0], b2[0])
        inter_y1 = max(b1[1], b2[1])
        inter_x2 = min(b1[2], b2[2])
        inter_y2 = min(b1[3], b2[3])
        inter_w = max(0, inter_x2 - inter_x1)
        inter_h = max(0, inter_y2 - inter_y1)
        inter_area = inter_w * inter_h
        area1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        area2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
        union_area = area1 + area2 - inter_area
        iou = inter_area / union_area if union_area > 0 else 0.0
        
        diou_penalty = rho2 / c2_diag if c2_diag > 0 else 0.0
        diou_score = iou - diou_penalty
        
        print(f"Pair {i+1}: Connector A (conf={c1[2]:.3f}) vs Connector B (conf={c2[2]:.3f})")
        print(f"  Box A: {b1} | Center: ({cx1:.1f}, {cy1:.1f})")
        print(f"  Box B: {b2} | Center: ({cx2:.1f}, {cy2:.1f})")
        print(f"  Geometric Center Distance (ρ):        {rho:.2f} px")
        print(f"  Enclosing Box Diagonal (c):            {c_diag:.2f} px")
        print(f"  Standard IoU Overlap:                  {iou:.4f}")
        print(f"  DIoU Distance Penalty (ρ²/c²):         {diou_penalty:.4f}")
        print(f"  DIoU Metric Score (IoU - ρ²/c²):       {diou_score:.4f}")
        if diou_score < 0.45:
            print("  Decision Status:                       ✅ PRESERVED by DIoU-NMS (distinct spatial center prevents false suppression)")
        else:
            print("  Decision Status:                       ⚠️ High overlap detected")
        print("-" * 75)

print("\n" + "=" * 88)
print("✅ NMS Results inspection completed successfully!")
print("=" * 88)
